# 05 — DB-First Topological Map

Visualise the active DB-first leaf graph rebuilt from OpenClaw traces.

Focus:
- active `training_data` build metadata
- leaf-node frequency and session coverage
- weighted `next` transitions between leaves
- sequence-length distribution for notebook-first training analysis


In [9]:
import {
  getActiveTrainingBuildId,
  listActiveSessionSequences,
  listActiveToolLeafEdgesNext,
  listActiveToolLeafNodes,
} from "../src/training-data/rebuild.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const kv = await Deno.openKv(DB_PATH);
const buildId = await getActiveTrainingBuildId(kv);
if (!buildId) {
  throw new Error(
    `No active DB-first training build found in ${DB_PATH}. Run 'deno task cli init ${VAULT_PATH}' first.`,
  );
}

const nodes = await listActiveToolLeafNodes(kv);
const edges = await listActiveToolLeafEdgesNext(kv);
const sequences = await listActiveSessionSequences(kv);

console.log(`Active build: ${buildId}`);
console.log(`Leaf nodes: ${nodes.length}`);
console.log(`Leaf transitions: ${edges.length}`);
console.log(`Session sequences: ${sequences.length}`);


Active build: 2b103f4f-a717-44f2-83ef-305eb0149166
Leaf nodes: 107
Leaf transitions: 994
Session sequences: 231
Leaf nodes: 107
Leaf transitions: 994
Session sequences: 231


In [10]:
const incoming = new Map<string, number>();
const outgoing = new Map<string, number>();

for (const node of nodes) {
  incoming.set(node.leafKey, 0);
  outgoing.set(node.leafKey, 0);
}

for (const edge of edges) {
  outgoing.set(edge.fromLeaf, (outgoing.get(edge.fromLeaf) ?? 0) + edge.weight);
  incoming.set(edge.toLeaf, (incoming.get(edge.toLeaf) ?? 0) + edge.weight);
}

const nodeRows = nodes.map((node) => ({
  leafKey: node.leafKey,
  toolRoot: node.toolRoot,
  level: node.level,
  isFallback: node.isFallback,
  totalOccurrences: node.totalOccurrences,
  uniqueSessions: node.uniqueSessions,
  incomingWeight: incoming.get(node.leafKey) ?? 0,
  outgoingWeight: outgoing.get(node.leafKey) ?? 0,
  topLevelShare: node.totalOccurrences === 0
    ? 0
    : node.topLevelOccurrences / node.totalOccurrences,
})).sort((left, right) =>
  right.totalOccurrences - left.totalOccurrences ||
  right.outgoingWeight - left.outgoingWeight ||
  left.leafKey.localeCompare(right.leafKey)
);

const edgeRows = edges.map((edge) => ({
  ...edge,
  topLevelShare: edge.weight === 0 ? 0 : edge.topLevelWeight / edge.weight,
})).sort((left, right) =>
  right.weight - left.weight ||
  left.fromLeaf.localeCompare(right.fromLeaf) ||
  left.toLeaf.localeCompare(right.toLeaf)
);

const lengthRows = sequences.map((seq) => ({
  sessionId: seq.sessionId,
  sessionKind: seq.sessionKind,
  callCount: seq.callCount,
})).sort((left, right) => right.callCount - left.callCount);

console.log("\nTop leaves:");
console.table(nodeRows.slice(0, 15).map((row) => ({
  leafKey: row.leafKey,
  totalOccurrences: row.totalOccurrences,
  uniqueSessions: row.uniqueSessions,
  incomingWeight: row.incomingWeight,
  outgoingWeight: row.outgoingWeight,
  topLevelShare: row.topLevelShare.toFixed(2),
}))); 

console.log("\nTop transitions:");
console.table(edgeRows.slice(0, 15).map((row) => ({
  fromLeaf: row.fromLeaf,
  toLeaf: row.toLeaf,
  weight: row.weight,
  topLevelWeight: row.topLevelWeight,
  subagentWeight: row.subagentWeight,
}))); 

console.log("\nLongest sequences:");
console.table(lengthRows.slice(0, 10));



Top leaves:
┌───────┬─────────────────────────────────────┬──────────────────┬────────────────┬────────────────┬────────────────┬───────────────┐
│ (idx) │ leafKey                             │ totalOccurrences │ uniqueSessions │ incomingWeight │ outgoingWeight │ topLevelShare │
├───────┼─────────────────────────────────────┼──────────────────┼────────────────┼────────────────┼────────────────┼───────────────┤
│     0 │ "tool.exec.other_shell"             │             3372 │            177 │           3331 │           3332 │ "1.00"        │
│     1 │ "tool.exec.inspect_fs_text"         │              962 │             70 │            959 │            951 │ "1.00"        │
│     2 │ "tool.exec.network_http"            │              655 │            135 │            553 │            644 │ "1.00"        │
│     3 │ "tool.process.poll"                 │              524 │            135 │            524 │            448 │ "1.00"        │
│     4 │ "tool.exec.runtime_build"           │  

In [11]:
const leafSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Leaf nodes by occurrence and transition pressure",
  "width": 760,
  "height": 360,
  "data": { "values": nodeRows.slice(0, 30) },
  "mark": { "type": "circle", "opacity": 0.85 },
  "encoding": {
    "x": { "field": "incomingWeight", "type": "quantitative", "title": "Incoming transition weight" },
    "y": { "field": "outgoingWeight", "type": "quantitative", "title": "Outgoing transition weight" },
    "size": { "field": "totalOccurrences", "type": "quantitative", "title": "Occurrences", "scale": { "range": [120, 1600] } },
    "color": { "field": "topLevelShare", "type": "quantitative", "title": "Top-level share", "scale": { "scheme": "viridis" } },
    "tooltip": [
      { "field": "leafKey" },
      { "field": "toolRoot" },
      { "field": "totalOccurrences" },
      { "field": "uniqueSessions" },
      { "field": "incomingWeight" },
      { "field": "outgoingWeight" },
      { "field": "topLevelShare", "format": ".2f" }
    ]
  }
};

const edgeSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Top weighted next transitions",
  "width": 760,
  "height": 320,
  "data": { "values": edgeRows.slice(0, 20) },
  "mark": "bar",
  "encoding": {
    "x": { "field": "weight", "type": "quantitative", "title": "Transition weight" },
    "y": {
      "field": "label",
      "type": "nominal",
      "sort": "-x",
      "title": "Transition"
    },
    "color": { "field": "topLevelShare", "type": "quantitative", "title": "Top-level share", "scale": { "scheme": "plasma" } },
    "tooltip": [
      { "field": "fromLeaf" },
      { "field": "toLeaf" },
      { "field": "weight" },
      { "field": "topLevelWeight" },
      { "field": "subagentWeight" }
    ]
  },
  "transform": [
    {
      "calculate": "datum.fromLeaf + ' → ' + datum.toLeaf",
      "as": "label"
    }
  ]
};

const sequenceSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Session sequence length distribution",
  "width": 760,
  "height": 260,
  "data": { "values": lengthRows },
  "mark": "bar",
  "encoding": {
    "x": { "field": "callCount", "type": "quantitative", "bin": true, "title": "Calls per session" },
    "y": { "aggregate": "count", "type": "quantitative", "title": "Sessions" },
    "color": { "field": "sessionKind", "type": "nominal", "title": "Session kind" },
    "tooltip": [
      { "aggregate": "count", "type": "quantitative", "title": "Sessions" },
      { "field": "sessionKind", "type": "nominal" }
    ]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": leafSpec }, { raw: true });
await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": edgeSpec }, { raw: true });
await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": sequenceSpec }, { raw: true });


In [12]:
kv.close();
console.log("Done");


Done
